# 09 — Public external clinical-image training experiment
Run all cells in a fresh Colab T4 runtime. Uses the **public Zenodo Fitzpatrick17k image archive**, not email-only access. The archive is 1,422,904,980 bytes (about 1.4 GB). It contains raw renamed Fitzpatrick17k images; they are not all cleaned examples. Therefore this notebook intersects the authors' corrected metadata **training partition**, matching diagnoses/skin-type annotations, and screens duplicates against the existing project splits. External corrected validation/test partitions are not used for training. Atlas diagnoses and image-based skin annotations are not claimed biopsy-confirmed or clinical ground truth.

Sources: https://derm.cs.sfu.ca/critique/ and https://zenodo.org/records/11101338 (image archive), https://zenodo.org/records/12739457 (corrected metadata). Cite Abhishek, Jain and Hamarneh, Scientific Data 2025, DOI 10.1038/s41597-025-04382-5, and the original Fitzpatrick17k authors Groh et al. 2021/2022. Use noncommercial research terms and preserve the original image attribution/license obligations; the metadata license is not a blanket relicense of all source images.

This experiment can increase available SCC and recorded skin-type training support; it does not guarantee target attainment or independent subgroup reliability. Existing app checkpoint and test manifests remain untouched.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import shutil
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display

print('TensorFlow:', tf.__version__)
print('Keras:', getattr(tf.keras, '__version__', 'bundled with TensorFlow'))
gpus = tf.config.list_physical_devices('GPU')
print('Available GPUs:', gpus)
if not gpus:
    raise RuntimeError('Enable the T4 GPU before continuing.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        print('GPU already initialized; keeping its memory configuration.')

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
HEAD_EPOCHS = 6
FINE_TUNE_EPOCHS = 8
CLASS_NAMES = ['BCC', 'Melanoma', 'SCC']
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
PROJECT_ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
SPLIT_ROOT = PROJECT_ROOT / 'Datasets' / 'Splits'
MODEL_ROOT = PROJECT_ROOT / 'Models' / 'Transfer_Learning'
REPORT_ROOT = PROJECT_ROOT / 'Reports' / 'Training'
LOCAL_ROOT = Path('/content/skin_cancer_training_images')
for folder in [MODEL_ROOT, REPORT_ROOT, LOCAL_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

def write_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)
frames = {}
manifest_hashes = {}
for split in ['train', 'validation', 'test']:
    path = SPLIT_ROOT / f'{split}_metadata.csv'
    if not path.is_file():
        raise FileNotFoundError(path)
    manifest_hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    frame = pd.read_csv(path, dtype={'patient_id': 'string', 'lesion_id': 'string'})
    required = {'dataset', 'image_id', 'class_label', 'image_path', 'image_hash', 'group_id', 'label_index', 'split'}
    if not required.issubset(frame.columns):
        raise ValueError(f'{split}: required split columns are missing.')
    if not frame['split'].eq(split).all():
        raise ValueError(f'{split}: incorrect split assignments.')
    if set(frame['class_label']) != set(CLASS_NAMES):
        raise ValueError(f'{split}: expected all three classes.')
    expected_labels = frame['class_label'].map(CLASS_TO_INDEX)
    if not expected_labels.eq(frame['label_index']).all():
        raise ValueError(f'{split}: class order differs from notebook 4.')
    frames[split] = frame

combined = pd.concat(frames.values(), ignore_index=True)
assert len(combined) == 2703
assert not combined.duplicated(['dataset', 'image_id']).any()
assert combined.groupby('group_id')['split'].nunique().max() == 1
assert combined.groupby('image_hash')['split'].nunique().max() == 1
pad = combined[combined['dataset'] == 'PAD-UFES-20']
assert pad['patient_id'].notna().all()
assert pad.groupby('patient_id')['split'].nunique().max() == 1
lesions = combined.dropna(subset=['lesion_id'])
assert lesions.groupby(['dataset', 'lesion_id'])['split'].nunique().max() == 1

train_frame = frames['train'].copy()
val_frame = frames['validation'].copy()
train_counts = train_frame['label_index'].value_counts().reindex(range(3))
CLASS_WEIGHTS = {
    index: float(len(train_frame) / (3 * train_counts[index]))
    for index in range(3)
}
print('Training records:', len(train_frame))
print('Validation records:', len(val_frame))
print('Test records reserved for notebook 6:', len(frames['test']))
print('Class order:', CLASS_TO_INDEX)
print('Training-only class weights:', CLASS_WEIGHTS)
display(pd.crosstab(combined['split'], combined['class_label']))


if not hasattr(tf.keras.losses, 'CategoricalFocalCrossentropy'):
    raise RuntimeError('This runtime needs a TensorFlow/Keras version with CategoricalFocalCrossentropy. Do not downgrade to the old app requirements.')

baseline_folder = MODEL_ROOT / 'exp02_65b8c20800f3' / 'ResNet50'
baseline = json.loads((baseline_folder / 'metadata.json').read_text())
if not baseline.get('training_complete') or baseline['manifest_hashes'] != manifest_hashes:
    raise ValueError('Baseline incomplete or split manifests changed.')


EXTERNAL_ROOT = PROJECT_ROOT / 'Datasets' / 'External_Fitzpatrick17k'
EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_PACKAGE = EXTERNAL_ROOT / 'images'
LOCAL_PACKAGE.mkdir(exist_ok=True)
# Image archive and corrected metadata download automatically in the following cell.
EXTERNAL_REPORTS = PROJECT_ROOT / 'Reports' / 'External_Data_Preparation'
EXTERNAL_REPORTS.mkdir(parents=True, exist_ok=True)


## Download verified public archive and corrected metadata
Only exact BCC, Melanoma and SCC diagnoses with agreeing original phototype annotations are eligible. Restrict to the corrected metadata's training partition and retain matching diagnosis/phototype labels. Validate published file checksums and each extracted image's original MD5. Archive paths are never extracted blindly. Downloads and verification reports remain on private Drive, outside GitHub.


In [ ]:
import urllib.request
import zipfile
from PIL import Image
import io

METADATA_URL='https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv'
CLEAN_URL='https://zenodo.org/records/12739457/files/Fitzpatrick17k-C.csv?download=1'
ARCHIVE_URL='https://zenodo.org/records/11101338/files/Fitzpatrick17k_CategorizedAbbrvs.zip?download=1'
ARCHIVE_MD5='8f2640a6a45bbaf5c9b776d2d68fccc6'
CLEAN_MD5='028e4fc67f64526b264a92d5b0e57909'

def file_md5(path):
    h=hashlib.md5()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()

def download_verified(url,destination,expected_md5=None):
    destination=Path(destination)
    if destination.is_file():
        if expected_md5 is None or file_md5(destination)==expected_md5:
            print('Using verified cached file:',destination.name)
            return
        raise ValueError('Existing file checksum differs: '+str(destination)+'; preserve/inspect it before retrying.')
    partial=destination.with_suffix(destination.suffix+'.part')
    with urllib.request.urlopen(url,timeout=90) as response,partial.open('wb') as output:
        written=0;last_report=0
        while True:
            block=response.read(4*1024*1024)
            if not block:break
            output.write(block);written+=len(block)
            if written-last_report>=50*1024*1024:
                print(destination.name,round(written/1024/1024),'MB downloaded',flush=True);last_report=written
    if expected_md5 and file_md5(partial)!=expected_md5:
        raise ValueError('Downloaded file checksum differs; retained partial file: '+str(partial))
    partial.replace(destination)
    print('Download verified:',destination.name)

metadata_path=EXTERNAL_ROOT/'fitzpatrick17k.csv'
clean_path=EXTERNAL_ROOT/'Fitzpatrick17k-C.csv'
archive_path=EXTERNAL_ROOT/'Fitzpatrick17k_CategorizedAbbrvs.zip'
download_verified(METADATA_URL,metadata_path)
download_verified(CLEAN_URL,clean_path,CLEAN_MD5)
f17=pd.read_csv(metadata_path)
clean=pd.read_csv(clean_path,dtype={'md5hash':str})
assert not clean.md5hash.duplicated().any()
def select_training_records(f17,clean):
    label_map={'basal cell carcinoma':'BCC','melanoma':'Melanoma','squamous cell carcinoma':'SCC'}
    first=pd.to_numeric(f17.fitzpatrick_scale,errors='coerce')
    second=pd.to_numeric(f17.fitzpatrick_centaur,errors='coerce')
    eligible=f17.loc[f17.label.isin(label_map)&first.isin(range(1,7))&first.eq(second)].copy()
    eligible=eligible.drop_duplicates('md5hash')
    clean_train=clean.loc[clean.partition.eq('train'),['md5hash','label','fst','new_img_name']].rename(columns={'label':'corrected_label'})
    eligible=eligible.merge(clean_train,on='md5hash',how='inner',validate='one_to_one')
    eligible=eligible.loc[eligible.label.eq(eligible.corrected_label)&pd.to_numeric(eligible.fitzpatrick_scale).eq(pd.to_numeric(eligible.fst))].copy()
    eligible['class_label']=eligible.label.map(label_map)
    return eligible.sort_values('md5hash').reset_index(drop=True)

eligible=select_training_records(f17,clean)
if eligible.empty:raise ValueError('No eligible corrected training examples.')
eligible.to_csv(EXTERNAL_REPORTS/'eligible_source_records.csv',index=False)
display(pd.crosstab(eligible.class_label,eligible.fitzpatrick_scale))
download_verified(ARCHIVE_URL,archive_path,ARCHIVE_MD5)
def image_fingerprints(path):
    raw=Path(path).read_bytes()
    with Image.open(io.BytesIO(raw)) as im:
        if im.width * im.height > 20_000_000: raise ValueError('Image exceeds pixel limit')
        im=im.convert('RGB')
        pixel_hash=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
        small=np.asarray(im.convert('L').resize((9,8)),dtype=np.int16)
        bits=(small[:,1:]>small[:,:-1]).reshape(-1)
        dhash=sum(int(bit)<<i for i,bit in enumerate(bits))
    return hashlib.sha256(raw).hexdigest(),pixel_hash,dhash

records=[]
with zipfile.ZipFile(archive_path) as z:
    members={}
    for member in z.infolist():
        if member.is_dir():continue
        members.setdefault(Path(member.filename).name,[]).append(member)
    for row in eligible.to_dict('records'):
        target=EXTERNAL_ROOT/(row['md5hash']+'.jpg')
        try:
            if not target.is_file():
                matches=members.get(row['new_img_name'],[])
                if len(matches)!=1:raise ValueError('Missing or ambiguous archive member: '+row['new_img_name'])
                member=matches[0]
                if member.file_size>20_000_000:raise ValueError('Archive image exceeds 20 MB')
                raw=z.read(member)
                if hashlib.md5(raw).hexdigest()!=row['md5hash']:raise ValueError('Image bytes differ from original metadata MD5')
                temporary=target.with_suffix('.tmp');temporary.write_bytes(raw)
                image_fingerprints(temporary)
                temporary.replace(target)
            if file_md5(target)!=row['md5hash']:raise ValueError('Cached image MD5 mismatch')
            byte_hash,pixel_hash,dhash=image_fingerprints(target)
            row.update(image_path=str(target),image_hash=byte_hash,pixel_hash=pixel_hash,dhash=str(dhash),download_status='ok',download_error='')
        except Exception as exc:
            row.update(download_status='failed',download_error=str(exc))
        records.append(row)
downloads=pd.DataFrame(records)
downloads.to_csv(EXTERNAL_REPORTS/'download_results.csv',index=False)
available=downloads.loc[downloads.download_status.eq('ok')].copy()
print('Verified eligible training images:',len(available),'Failed:',len(downloads)-len(available))
if available.empty:raise RuntimeError('No images passed integrity checks; see download_results.csv. No training started.')


## Exclude overlap and duplicate candidates
Compare against **every existing split** using identifiers, byte hashes, decoded pixel hashes, and a conservative 64-bit difference-hash distance of at most four. Exclude candidates rather than decide manually that a possible duplicate is safe. This screen is not proof that all same-patient or near-duplicate images are found: external patient IDs are unavailable. No external records are used as a supposedly independent test set.


In [ ]:
existing=[]
for n,row in enumerate(combined.itertuples(index=False)):
    byte_hash,pixel_hash,dhash=image_fingerprints(row.image_path)
    existing.append((byte_hash,pixel_hash,dhash))
    if (n+1)%500==0: print('Existing images screened:',n+1)
byte_seen={x[0] for x in existing};pixel_seen={x[1] for x in existing};perceptual_seen=[x[2] for x in existing]
existing_ids={Path(str(x)).stem for x in combined.image_id}
accepted=[];rejected=[]
for row in available.sort_values('md5hash').to_dict('records'):
    dh=int(row['dhash'])
    reason=None
    if row['md5hash'] in existing_ids: reason='identifier_overlap'
    elif row['image_hash'] in byte_seen: reason='byte_duplicate'
    elif row['pixel_hash'] in pixel_seen: reason='pixel_duplicate'
    if reason is None and any((dh ^ other).bit_count()<=4 for other in perceptual_seen): reason='possible_near_duplicate'
    if reason:
        rejected.append({**row,'exclusion_reason':reason});continue
    accepted.append(row)
    byte_seen.add(row['image_hash']);pixel_seen.add(row['pixel_hash']);perceptual_seen.append(dh)
external=pd.DataFrame(accepted)
pd.DataFrame(rejected).to_csv(EXTERNAL_REPORTS/'excluded_duplicates.csv',index=False)
if external.empty: raise RuntimeError('No external images survived overlap screening.')
external['dataset']='Fitzpatrick17k'
external['image_id']=external.md5hash+'.jpg'
external['label_index']=external.class_label.map(CLASS_TO_INDEX)
external['split']='train'
external['patient_id']=pd.NA
external['lesion_id']=pd.NA
external['group_id']='F17K_'+external.md5hash
external['original_label']=external.label
external['biopsy_confirmed']=False
external['diagnosis_method']='original atlas label; not verified biopsy'
external['fitzpatrick']=external.fitzpatrick_scale
external['fitzpatrick_source']='two agreeing image-based annotations; not clinically verified'
external.to_csv(EXTERNAL_REPORTS/'external_training_manifest.csv',index=False)
# Require actual additions addressing both tasks before consuming GPU time.
if (external.class_label=='SCC').sum()<20:
    raise RuntimeError('Fewer than 20 additional SCC images survived; inspect source failures.')
for label in CLASS_NAMES:
    if not ((external.class_label==label)&external.fitzpatrick.isin([4,5,6])).any():
        raise RuntimeError('No usable recorded IV–VI external examples for '+label)
train_frame=pd.concat([frames['train'],external],ignore_index=True)
val_frame=frames['validation'].copy()
CLASS_WEIGHTS={i:float(len(train_frame)/(3*(train_frame.label_index==i).sum())) for i in range(3)}
FOCAL_ALPHA=[CLASS_WEIGHTS[i] for i in range(3)]
EXPERIMENT_SETTINGS=dict(experiment='experiment_07_zenodo_corrected_train',seed=SEED,
    image_size=list(IMAGE_SIZE), external_manifest_sha256=hashlib.sha256((EXTERNAL_REPORTS/'external_training_manifest.csv').read_bytes()).hexdigest(),
    external_metadata_sha256=hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
    external_corrected_metadata_sha256=hashlib.sha256(clean_path.read_bytes()).hexdigest(),
    external_archive_md5=ARCHIVE_MD5, external_partition='corrected train only',
    manifest_hashes=manifest_hashes, loss='weighted categorical crossentropy via focal gamma zero',
    alpha=FOCAL_ALPHA, external_patient_ids_available=False, independent_skin_type_evaluation=False)
RUN_ID='exp07_'+hashlib.sha256(json.dumps(EXPERIMENT_SETTINGS,sort_keys=True).encode()).hexdigest()[:12]
RUN_MODELS=MODEL_ROOT/RUN_ID;RUN_REPORTS=REPORT_ROOT/RUN_ID
RUN_MODELS.mkdir(parents=True,exist_ok=True);RUN_REPORTS.mkdir(parents=True,exist_ok=True)
train_frame.to_csv(RUN_REPORTS/'augmented_training_manifest.csv',index=False)
write_json(RUN_REPORTS/'training_settings.json',EXPERIMENT_SETTINGS)
HEAD_EPOCHS=12;FINE_TUNE_EPOCHS=25
print('External training images retained:',len(external))
display(pd.crosstab(external.class_label,external.fitzpatrick))
print('Training run:',RUN_ID,'Validation unchanged:',len(val_frame))


## Image pipeline and training
Uses the existing 224×224 RGB ResNet50 preprocessing contract, train-only augmentation, natural sampling, training-only class weights, and validation macro F1 checkpoint selection. The external-data change is experimental; atlas-label uncertainty and source differences remain limitations. Test results do not choose the checkpoint.


In [ ]:
# Copy only training and validation images; test images are not loaded.
for split, frame in [('train', train_frame), ('validation', val_frame)]:
    local_paths = []
    for position, row in enumerate(frame.itertuples(index=False)):
        source = Path(row.image_path)
        if not source.is_file():
            raise FileNotFoundError(source)
        if source.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp'}:
            raise ValueError(f'Unsupported training image format: {source}')
        key = hashlib.sha256(str(source).encode()).hexdigest()
        destination = LOCAL_ROOT / (key + source.suffix.lower())
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            temporary = destination.with_suffix(destination.suffix + '.tmp')
            shutil.copyfile(source, temporary)
            temporary.replace(destination)
        local_paths.append(str(destination))
        if (position + 1) % 250 == 0:
            print(f'{split}: {position + 1}/{len(frame)} copied')
    frame['local_path'] = local_paths
    print(f'{split}: local images ready')

def decode_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, method='bilinear')
    return image, label

def raw_dataset(frame):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['local_path'].to_numpy(dtype=str),
        frame['label_index'].to_numpy(dtype=np.int32),
    ))
    return ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE).cache()

RAW_TRAIN = raw_dataset(train_frame)
RAW_VAL = raw_dataset(val_frame)
# Fully populate caches once, before model training.
for _ in RAW_TRAIN.batch(BATCH_SIZE):
    pass
for _ in RAW_VAL.batch(BATCH_SIZE):
    pass
VAL_LABELS = val_frame['label_index'].to_numpy(dtype=np.int32)
images, labels = next(iter(RAW_TRAIN.batch(6)))
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for axis, image, label in zip(axes.flat, images.numpy(), labels.numpy()):
    axis.imshow(np.clip(image, 0, 255).astype('uint8'))
    axis.set_title(CLASS_NAMES[int(label)])
    axis.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
MODEL_SPECS = {
    'MobileNetV2': (tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input, 'mobilenet_v2'),
    'EfficientNetB0': (tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, 'efficientnet_b0'),
    'ResNet50': (tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input, 'resnet50'),
}

def model_datasets(name):
    preprocess = MODEL_SPECS[name][1]
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
        tf.keras.layers.RandomRotation(0.05, fill_mode='reflect', seed=SEED + 1),
        tf.keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED + 2),
        tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255), seed=SEED + 3),
    ])
    def prepare_training(images, labels):
        images = augmentation(images, training=True)
        return preprocess(tf.clip_by_value(images, 0.0, 255.0)), tf.one_hot(labels, depth=3)
    def prepare_validation(images, labels):
        return preprocess(images), tf.one_hot(labels, depth=3)
    train_ds = RAW_TRAIN.shuffle(len(train_frame), seed=SEED, reshuffle_each_iteration=True)
    train_ds = train_ds.batch(BATCH_SIZE).map(prepare_training, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    val_ds = RAW_VAL.batch(BATCH_SIZE).map(prepare_validation, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds

def classification_scores(labels, probabilities):
    predicted = probabilities.argmax(axis=1)
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (labels, predicted), 1)
    true_positive = np.diag(matrix).astype(float)
    precision = np.divide(true_positive, matrix.sum(axis=0), out=np.zeros(3), where=matrix.sum(axis=0) != 0)
    recall = np.divide(true_positive, matrix.sum(axis=1), out=np.zeros(3), where=matrix.sum(axis=1) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(3), where=(precision + recall) != 0)
    return float(f1.mean()), float((labels == predicted).mean())

def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=FOCAL_ALPHA, gamma=0.0),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')],
    )

class ValidationCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, name, val_ds, model_dir, report_dir):
        super().__init__()
        self.name = name
        self.val_ds = val_ds
        self.model_dir = model_dir
        self.report_dir = report_dir
        self.best = -1.0
        self.best_accuracy = None
        self.best_phase = None
        self.best_epoch = None
        self.phase = 'head'
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        probabilities = self.model.predict(self.val_ds, verbose=0)
        macro_f1, accuracy = classification_scores(VAL_LABELS, probabilities)
        logs['val_macro_f1'] = macro_f1
        self.rows.append({'phase': self.phase, 'epoch': epoch + 1, **{key: float(value) for key, value in logs.items()}})
        pd.DataFrame(self.rows).to_csv(self.report_dir / 'training_history.csv', index=False)
        print(f' — validation macro F1: {macro_f1:.4f}')
        if macro_f1 > self.best:
            self.best = macro_f1
            self.best_accuracy = accuracy
            self.best_phase = self.phase
            self.best_epoch = epoch + 1
            self.model.save(self.model_dir / 'best.keras')
            self.model.save_weights(self.model_dir / 'best.weights.h5')
            write_json(self.model_dir / 'checkpoint.json', {
                'validation_macro_f1': macro_f1, 'validation_accuracy': accuracy,
                'phase': self.phase, 'epoch': epoch + 1, 'class_names': CLASS_NAMES,
            })
            print('Best checkpoint saved to Drive.')


def reduce_lr_callback():
    return tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_macro_f1', mode='max', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1,
    )

def stop_callback():
    return tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=6, restore_best_weights=False)

def train_one_model(name):
    if name != 'ResNet50':
        raise ValueError('This bounded experiment is for ResNet50 only.')
    model_dir = RUN_MODELS / name
    report_dir = RUN_REPORTS / name
    model_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = model_dir / 'metadata.json'
    if metadata_path.is_file():
        saved = json.loads(metadata_path.read_text())
        if saved.get('training_complete') and saved.get('run_id') == RUN_ID and (model_dir / 'best.keras').is_file():
            if hashlib.sha256((model_dir / 'best.keras').read_bytes()).hexdigest() != saved['model_sha256']:
                raise ValueError('Completed checkpoint checksum differs.')
            print(name, 'already complete for this run; skipping.')
            return saved
    # Never overwrite an incomplete attempt: retain its checkpoints/history.
    if (model_dir / 'best.keras').exists() or (report_dir / 'training_history.csv').exists():
        stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
        model_dir.rename(RUN_MODELS / f'{name}_interrupted_{stamp}')
        report_dir.rename(RUN_REPORTS / f'{name}_interrupted_{stamp}')
        model_dir.mkdir()
        report_dir.mkdir()

    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    train_ds, val_ds = model_datasets(name)
    constructor, _, preprocess_name = MODEL_SPECS[name]
    base = constructor(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='preprocessed_image')
    features = base(inputs, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)
    features = tf.keras.layers.Dense(128, activation='relu', name='research_head')(features)
    features = tf.keras.layers.Dropout(0.30, seed=SEED)(features)
    outputs = tf.keras.layers.Dense(3, activation='softmax', name='class_scores')(features)
    model = tf.keras.Model(inputs, outputs, name=name + '_skin_classifier')
    monitor = ValidationCheckpoint(name, val_ds, model_dir, report_dir)
    started = time.time()

    print('\nTRAINING CLASSIFICATION HEAD:', name)
    compile_model(model, 0.001)
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              callbacks=[monitor, reduce_lr_callback(), stop_callback()], verbose=2)
    model.load_weights(model_dir / 'best.weights.h5')

    print('\nFINE-TUNING:', name)
    base.trainable = True
    cutoff = int(len(base.layers) * 0.50)
    for index, layer in enumerate(base.layers):
        layer.trainable = index >= cutoff and not isinstance(layer, tf.keras.layers.BatchNormalization)
    compile_model(model, 0.00001)
    monitor.phase = 'fine_tune'
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=[monitor, reduce_lr_callback(), stop_callback()], verbose=2)

    # Validate serialization: loaded best model must reproduce its weights.
    model.load_weights(model_dir / 'best.weights.h5')
    reference_batch, _ = next(iter(val_ds))
    reference_scores = model.predict(reference_batch, verbose=0)
    loaded = tf.keras.models.load_model(model_dir / 'best.keras', compile=False)
    reloaded_scores = loaded.predict(reference_batch, verbose=0)
    np.testing.assert_allclose(reference_scores, reloaded_scores, rtol=1e-4, atol=1e-5)
    np.savez(report_dir / 'validation_reference.npz', scores=reloaded_scores, labels=VAL_LABELS[:len(reloaded_scores)])
    val_frame.head(len(reloaded_scores)).to_csv(report_dir / 'validation_reference_images.csv', index=False)

    # Full validation diagnostics for the saved best checkpoint.
    from sklearn.metrics import classification_report, confusion_matrix
    validation_scores = loaded.predict(val_ds, verbose=0)
    predicted = validation_scores.argmax(axis=1)
    report = classification_report(
        VAL_LABELS, predicted, labels=[0, 1, 2],
        target_names=CLASS_NAMES, output_dict=True, zero_division=0,
    )
    class_report = pd.DataFrame([
        {'class_label': label, **report[label]} for label in CLASS_NAMES
    ])
    class_report.to_csv(report_dir / 'validation_per_class_metrics.csv', index=False)
    display(class_report)
    pd.DataFrame(confusion_matrix(VAL_LABELS, predicted, labels=[0, 1, 2]),
                 index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        report_dir / 'validation_confusion_matrix.csv')
    predictions = val_frame.drop(columns=['local_path'], errors='ignore').copy()
    predictions['predicted_class'] = [CLASS_NAMES[i] for i in predicted]
    for index, label in enumerate(CLASS_NAMES):
        predictions[f'score_{label}'] = validation_scores[:, index]
    predictions.to_csv(report_dir / 'validation_predictions.csv', index=False)
    predictions.loc[predictions['class_label'] != predictions['predicted_class']].to_csv(
        report_dir / 'validation_errors.csv', index=False)
    best_f1, best_accuracy = classification_scores(VAL_LABELS, validation_scores)
    np.testing.assert_allclose(best_f1, monitor.best, atol=1e-8)
    np.testing.assert_allclose(best_accuracy, monitor.best_accuracy, atol=1e-8)

    metadata = {
        'model_name': name, 'run_id': RUN_ID, 'training_complete': True,
        'model_file': str(model_dir / 'best.keras'),
        'model_sha256': hashlib.sha256((model_dir / 'best.keras').read_bytes()).hexdigest(),
        'class_names': CLASS_NAMES, 'class_to_index': CLASS_TO_INDEX,
        'image_size': list(IMAGE_SIZE), 'color_mode': 'RGB',
        'resize': 'tf.image.resize bilinear; stretch to 224x224',
        'decode': 'tf.io.decode_image channels=3; no EXIF orientation transform',
        'preprocessing': preprocess_name, 'preprocessing_location': 'outside saved model',
        'tensorflow_version': tf.__version__,
        'keras_version': getattr(tf.keras, '__version__', None),
        'manifest_hashes': manifest_hashes, 'class_weights': None,
        'focal_alpha': FOCAL_ALPHA, 'focal_gamma': 0.0,
        'selection_metric': 'validation_macro_f1',
        'validation_macro_f1': monitor.best,
        'validation_accuracy': monitor.best_accuracy,
        'best_phase': monitor.best_phase, 'best_epoch': monitor.best_epoch,
        'training_seconds': round(time.time() - started, 2),
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'experiment': 'experiment_07_zenodo_corrected_train',
        'experiment_settings': EXPERIMENT_SETTINGS,
        'original_test_results_already_inspected': True,
        'test_images_used_for_gradient_updates': False,
        'checkpoint_selection_uses_validation_only': True,
    }
    write_json(metadata_path, metadata)
    history = pd.DataFrame(monitor.rows)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['loss'], label='train')
    axes[0].plot(history['val_loss'], label='validation')
    axes[0].set_title('Loss')
    axes[1].plot(history['accuracy'], label='train')
    axes[1].plot(history['val_accuracy'], label='validation')
    axes[1].set_title('Accuracy')
    axes[2].plot(history['val_macro_f1'], label='validation macro F1')
    axes[2].set_title('Validation macro F1')
    for axis in axes:
        axis.set_xlabel('Recorded epoch (head then fine-tuning)')
        axis.legend()
    fig.tight_layout()
    fig.savefig(report_dir / 'training_curves.png', dpi=160)
    plt.close(fig)
    del loaded, model, base, monitor, train_ds, val_ds
    tf.keras.backend.clear_session()
    gc.collect()
    print(name, 'COMPLETE. Best validation macro F1:', metadata['validation_macro_f1'])
    return metadata


In [ ]:
external_result=train_one_model('ResNet50')
display(pd.DataFrame([external_result])[[ 'model_name','validation_macro_f1','validation_accuracy','training_seconds']])
per_class=pd.read_csv(RUN_REPORTS/'ResNet50'/'validation_per_class_metrics.csv')
display(per_class)
previous=pd.read_csv(PROJECT_ROOT/'Reports'/'Training'/'exp06_7fe97493900f'/'ResNet50'/'validation_per_class_metrics.csv')
display(previous.merge(per_class,on='class_label',suffixes=('_experiment6','_external_data')))
print('Actual external-data experiment completed:',RUN_ID)
print('Validation macro F1 change:',external_result['validation_macro_f1']-0.7056333732653949)
print('Reports:',RUN_REPORTS)
print('No test evaluation, candidate replacement, or app integration performed.')
print('Representation increased only if the external manifest reports successful additions; fairness remains unestablished.')


## Interpretation and saving
Save the executed notebook to Drive and GitHub. Commit message: `Record external clinical-data training results`. Report the actual retained image counts and validation class metrics. Additional images do not guarantee target attainment. Do not use this notebook to claim independent patient separation for external data or proven darker-skin reliability. To establish the latter, a separately acquired representative real evaluation set is still required. Do not lower the requirements or select using repeatedly inspected test results.
